# 07. 영상에서 라벨링용 frame 이미지 추출\n\n이 notebook은 영상 파일 1개 또는 영상 파일들이 들어 있는 폴더를 입력으로 받아 Label Studio bbox 라벨링에 사용할 image frame을 저장합니다.\n\n기본 사용 방식은 `INTERVAL_SECONDS=2.0`처럼 몇 초마다 한 장을 저장하는 방식입니다. 영상 FPS는 OpenCV가 먼저 읽고, 그 FPS에 맞춰 실제 frame 간격을 계산합니다.\n\n안전 기본값은 실제 저장 cell이 꺼진 상태입니다. 먼저 preview로 예상 영상 수, 예상 frame 수, output 경로를 확인한 뒤 `RUN_EXTRACT=True`로 바꿔 실행하세요.\n\n핵심 용어:\n- `INPUT_PATH`: 영상 파일 1개 또는 영상들이 들어 있는 폴더입니다.\n- `RECURSIVE`: 폴더 입력일 때 하위 폴더까지 찾을지 정합니다. 기본값은 `False`라서 대상 폴더 바로 아래만 찾습니다.\n- `INTERVAL_SECONDS`: 몇 초마다 한 장을 저장할지 정합니다. 기본값은 2초입니다.\n- `OUT_DIR`: frame 이미지와 manifest를 저장할 별도 폴더입니다.\n- `manifest`: 저장된 frame이 원본 영상의 어느 frame/timestamp에서 왔는지 기록한 CSV/JSON입니다.\n

In [ ]:
from pathlib import Path\nimport sys\n\n# notebook을 repo 어느 위치에서 열어도 src package를 찾기 위한 bootstrap입니다.\nREPO_ROOT = Path.cwd().resolve()\nwhile REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():\n    REPO_ROOT = REPO_ROOT.parent\nif str(REPO_ROOT / "src") not in sys.path:\n    sys.path.insert(0, str(REPO_ROOT / "src"))\n\nfrom labelstudio_bbox_tools.video.frame_extract import extract_video_frames\n\nprint("repo:", REPO_ROOT)\nprint("video frame extraction module loaded")\n

In [ ]:
from pathlib import Path\n\n# ===== 사용자 설정 =====\n# 영상 파일 1개 또는 영상 파일들이 들어 있는 폴더를 지정하세요.\n# 예: INPUT_PATH = Path("/path/to/video.mp4")\n# 예: INPUT_PATH = Path("/path/to/video_folder")\nINPUT_PATH = Path("")\n\n# frame 이미지와 manifest를 저장할 별도 output root입니다.\n# 요청하신 4090 서버 기본값입니다. 다른 서버에서는 이 값을 바꾸면 됩니다.\nOUT_DIR = Path("/share_ssd/ltb/Users/ltb/label_studio/frames_export")\n\n# 폴더 입력일 때 False면 대상 폴더 바로 아래만 찾고, True면 하위 폴더까지 모두 찾습니다.\nRECURSIVE = False\n\n# 기본 추천 방식: 영상 FPS를 읽어서 N초마다 한 장을 저장합니다.\nINTERVAL_SECONDS = 2.0\n\n# 아래 두 방식은 특수한 경우에만 사용하세요. 사용할 때는 INTERVAL_SECONDS = None으로 바꿉니다.\nEVERY_N_FRAMES = None   # 예: 60이면 원본 frame 60장마다 한 장 저장\nTARGET_FPS = None       # 예: 0.5이면 결과 이미지를 초당 0.5장 정도로 저장\n\n# 긴 영상의 일부 구간만 테스트하고 싶을 때 사용합니다. None이면 제한 없음입니다.\nSTART_SECONDS = None\nEND_SECONDS = None\n\n# 처음 테스트할 때는 작은 값으로 제한해도 됩니다. None이면 제한 없음입니다.\nMAX_FRAMES_PER_VIDEO = None\nMAX_VIDEOS = None\n\n# bbox 라벨링용이면 jpg 95가 용량과 화질 균형이 좋습니다.\nIMAGE_FORMAT = "jpg"\nJPG_QUALITY = 95\n\n# True면 이미 같은 이름의 frame이 있을 때 덮어쓰지 않고 건너뜁니다.\nSKIP_EXISTING = True\n\n# 실제 저장 cell에서 이 값을 True로 바꿔야 frame 이미지가 저장됩니다.\nRUN_EXTRACT = False\nWRITE_MANIFEST = True\n

In [ ]:
# Preview cell입니다. 실제 이미지는 저장하지 않고 영상 metadata와 예상 frame 수만 확인합니다.\n# INPUT_PATH가 비어 있으면 여기서 멈춥니다.\nif not str(INPUT_PATH):\n    raise ValueError("INPUT_PATH에 영상 파일 또는 영상 폴더 경로를 넣어주세요.")\n\npreview = extract_video_frames(\n    input_path=INPUT_PATH,\n    out_dir=OUT_DIR,\n    recursive=RECURSIVE,\n    interval_seconds=INTERVAL_SECONDS,\n    every_n_frames=EVERY_N_FRAMES,\n    target_fps=TARGET_FPS,\n    start_seconds=START_SECONDS,\n    end_seconds=END_SECONDS,\n    max_frames_per_video=MAX_FRAMES_PER_VIDEO,\n    max_videos=MAX_VIDEOS,\n    image_format=IMAGE_FORMAT,\n    jpg_quality=JPG_QUALITY,\n    skip_existing=SKIP_EXISTING,\n    dry_run=True,\n    write_manifest=False,\n)\n\nprint("video count:", len(preview.videos))\nprint("planned frames:", preview.planned_frames)\nprint("out_dir:", preview.out_dir)\nfor item in preview.videos[:10]:\n    print({\n        "video": item.video_rel_path,\n        "fps": item.fps,\n        "duration_seconds": item.duration_seconds,\n        "planned_frames": item.planned_frames,\n        "output_dir": item.output_dir,\n        "error": item.error,\n    })\n

In [ ]:
# 실제 frame 저장 cell입니다.\n# preview 결과가 맞는지 확인한 뒤 RUN_EXTRACT=True로 바꾸고 실행하세요.\nif not RUN_EXTRACT:\n    print("RUN_EXTRACT=False 이므로 실제 frame 저장은 하지 않습니다.")\nelse:\n    result = extract_video_frames(\n        input_path=INPUT_PATH,\n        out_dir=OUT_DIR,\n        recursive=RECURSIVE,\n        interval_seconds=INTERVAL_SECONDS,\n        every_n_frames=EVERY_N_FRAMES,\n        target_fps=TARGET_FPS,\n        start_seconds=START_SECONDS,\n        end_seconds=END_SECONDS,\n        max_frames_per_video=MAX_FRAMES_PER_VIDEO,\n        max_videos=MAX_VIDEOS,\n        image_format=IMAGE_FORMAT,\n        jpg_quality=JPG_QUALITY,\n        skip_existing=SKIP_EXISTING,\n        dry_run=False,\n        write_manifest=WRITE_MANIFEST,\n    )\n    print("saved frames:", result.saved_frames)\n    print("skipped existing:", result.skipped_existing)\n    print("failed frames:", result.failed_frames)\n    print("manifest:", result.manifest_path)\n    print("summary:", result.summary_path)\n

In [ ]:
# 저장 후 결과 확인 cell입니다. RUN_EXTRACT=True로 실행한 뒤 사용하세요.\n# manifest CSV에는 원본 영상 경로, frame index, timestamp, 저장된 이미지 경로가 들어갑니다.\ntry:\n    result\nexcept NameError:\n    print("아직 실제 추출을 실행하지 않았습니다.")\nelse:\n    print(result.as_dict())\n